# Example of OpenAI Decisions 

You need a JSON file determining:
- model
- context
- input 
- type of output and the labels for each output

The file containing these information is: openai_decision_example/openai_decision_example.json

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI


def main():
    base_dir = Path("openai_decision_example.ipynb").resolve().parent

    # Load the API key from the .env file.
    load_dotenv(base_dir / ".env")

    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError(
            "OPENAI_API_KEY is missing or empty. "
            "Check the .env file in the script's directory."
        )

    # Read the request payload.
    payload_path = base_dir / "openai_decision_example.json"

    if not payload_path.is_file():
        raise FileNotFoundError(
            f"Request payload file not found: {payload_path}"
        )

    with payload_path.open(encoding="utf-8") as f:
        payload = json.load(f)

    client = OpenAI()

    # Send the request.
    decision = client.decisions.create(**payload)

    print("\n=== FULL RESPONSE ===")
    print(decision.model_dump_json(indent=2))

    # Display a summary for each question.
    print("\n=== RESULTS ===")

    for answer in decision.answers:
        print(f"\nQuestion: {answer.name}")

        if answer.type == "refusal":
            print("Evaluation refused.")

        elif answer.type == "choice":
            print(f"Selected category: {answer.choice}")
            print(f"Confidence: {answer.confidence}")

        elif answer.type == "predicate":
            print(f"Probability of Yes: {answer.probability:.3f}")

        elif answer.type == "score":
            print(f"Score: {answer.score:.3f}")
            print(f"Confidence: {answer.confidence}")

        else:
            print(f"Unhandled response type: {answer.type}")


if __name__ == "__main__":
    main()

# Expected output:
# === FULL RESPONSE ===
{
  "answers": [
    {
      "type": "choice",
      "name": "department",
      "choice": "fatturazione",
      "probabilities": [
        {
          "value": "fatturazione",
          "probability": 0.90
        },
        {
          "value": "supporto_tecnico",
          "probability": 0.07
        },
        {
          "value": "vendite",
          "probability": 0.03
        }
      ],
      "confidence": 0.85
    },
    {
      "type": "predicate",
      "name": "customer_is_frustrated",
      "probability": 0.95
    },
    {
      "type": "score",
      "name": "frustration",
      "score": 2.64,
      "probabilities": [
        {
          "value": 0,
          "label": "Calmo",
          "probability": 0.01
        },
        {
          "value": 1,
          "label": "Infastidito",
          "probability": 0.04
        },
        {
          "value": 2,
          "label": "Frustrato",
          "probability": 0.25
        },
        {
          "value": 3,
          "label": "Molto arrabbiato",
          "probability": 0.70
        }
      ],
      "confidence": 0.65
    }
  ]
}